References:
- `ChatGPT`,
- `https://www.kaggle.com/code/nghihuynh/cifar-10-mlp-cnn-resnet9-classifier`

# Cross-Library Setup: PyTorch and TensorFlow Imports for Deep Learning

In [64]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import random
from torchvision import models

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks, datasets
import numpy as np
import random
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# CIFAR10
## Pytorch CNN model trained from scratch


### Define the Residual Block

In [65]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        if self.downsample:
            residual = self.downsample(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.relu(out)
        return out

### Define the ResNet model

In [66]:
class ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet, self).__init__()
        self.conv = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
        self.bn = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_channels, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

        layers = [ResidualBlock(in_channels, out_channels, stride, downsample)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = nn.AdaptiveAvgPool2d((1, 1))(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


### Data preprocessing and augmentation

In [4]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

### Load CIFAR-10 dataset

In [5]:
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

100%|██████████| 170498071/170498071 [00:10<00:00, 15792072.15it/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified


### Device setup

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Hyperparameter search

In [7]:
def random_search(num_trials=10):
    best_accuracy = 0.0
    best_params = None

    for trial in range(num_trials):
        lr = random.choice([0.1, 0.01, 0.001])
        momentum = random.choice([0.9, 0.95, 0.99])
        weight_decay = random.choice([1e-4, 5e-4, 1e-3])

        print(f"Trial {trial + 1}: lr={lr}, momentum={momentum}, weight_decay={weight_decay}")

        model = ResNet().to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

        for epoch in range(5):
            model.train()
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            scheduler.step()

        correct = 0
        total = 0
        model.eval()
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f"Accuracy: {accuracy:.2f}%")

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_params = {'lr': lr, 'momentum': momentum, 'weight_decay': weight_decay}

    print(f"Best Accuracy: {best_accuracy:.2f}% with params: {best_params}")
    return best_params


### Perform random search

In [8]:
best_hyperparams = random_search()

Trial 1: lr=0.01, momentum=0.95, weight_decay=0.001
Accuracy: 76.30%
Trial 2: lr=0.001, momentum=0.99, weight_decay=0.0005
Accuracy: 77.06%
Trial 3: lr=0.001, momentum=0.95, weight_decay=0.0001
Accuracy: 78.70%
Trial 4: lr=0.01, momentum=0.95, weight_decay=0.0001
Accuracy: 77.38%
Trial 5: lr=0.01, momentum=0.95, weight_decay=0.0001
Accuracy: 77.14%
Trial 6: lr=0.01, momentum=0.9, weight_decay=0.001
Accuracy: 77.20%
Trial 7: lr=0.01, momentum=0.99, weight_decay=0.0005
Accuracy: 72.72%
Trial 8: lr=0.1, momentum=0.9, weight_decay=0.0001
Accuracy: 71.85%
Trial 9: lr=0.01, momentum=0.9, weight_decay=0.0005
Accuracy: 77.36%
Trial 10: lr=0.001, momentum=0.9, weight_decay=0.0001
Accuracy: 74.86%
Best Accuracy: 78.70% with params: {'lr': 0.001, 'momentum': 0.95, 'weight_decay': 0.0001}


### Train final model

In [9]:
print(f"Training final model with best hyperparameters: {best_hyperparams}")
model = ResNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=best_hyperparams['lr'], momentum=best_hyperparams['momentum'], weight_decay=best_hyperparams['weight_decay'])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

Training final model with best hyperparameters: {'lr': 0.001, 'momentum': 0.95, 'weight_decay': 0.0001}


### Training with early stopping

In [12]:
epochs = 20
patience = 5
best_loss = float('inf')
stopping_counter = 0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss:.4f}")

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        stopping_counter = 0
        torch.save(model.state_dict(), 'best_resnet_cifar10.pth')
    else:
        stopping_counter += 1

    if stopping_counter >= patience:
        print("Early stopping triggered")
        break

Epoch [1/20], Loss: 0.7160
Epoch [2/20], Loss: 0.6532
Epoch [3/20], Loss: 0.6081
Epoch [4/20], Loss: 0.5629
Epoch [5/20], Loss: 0.5257
Epoch [6/20], Loss: 0.5015
Epoch [7/20], Loss: 0.4745
Epoch [8/20], Loss: 0.3838
Epoch [9/20], Loss: 0.3592
Epoch [10/20], Loss: 0.3496
Epoch [11/20], Loss: 0.3467
Epoch [12/20], Loss: 0.3399
Epoch [13/20], Loss: 0.3285
Epoch [14/20], Loss: 0.3247
Epoch [15/20], Loss: 0.3190
Epoch [16/20], Loss: 0.3157
Epoch [17/20], Loss: 0.3119
Epoch [18/20], Loss: 0.3048
Epoch [19/20], Loss: 0.2986
Epoch [20/20], Loss: 0.3000


### Evaluate the Model

In [13]:
model.load_state_dict(torch.load('best_resnet_cifar10.pth'))
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy on the test set: {accuracy:.2f}%")


/tmp/ipykernel_23/3317349382.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_resnet_cifar10.pth'))


Accuracy on the test set: 88.94%


---
## Keras CNN model trained from scratch

### Define the Residual Block


In [51]:
class ResidualBlock(layers.Layer):
    def __init__(self, filters, stride=1, downsample=False):
        super(ResidualBlock, self).__init__()
        self.conv1 = layers.Conv2D(filters, kernel_size=3, strides=stride, padding='same', use_bias=False)
        self.bn1 = layers.BatchNormalization()
        self.relu = layers.ReLU()
        self.conv2 = layers.Conv2D(filters, kernel_size=3, strides=1, padding='same', use_bias=False)
        self.bn2 = layers.BatchNormalization()
        self.downsample = None
        
        if downsample:
            self.downsample = models.Sequential([
                layers.Conv2D(filters, kernel_size=1, strides=stride, use_bias=False),
                layers.BatchNormalization()
            ])
    
    def call(self, inputs, training=False):
        residual = inputs
        if self.downsample:
            residual = self.downsample(inputs, training=training)
        
        x = self.conv1(inputs)
        x = self.bn1(x, training=training)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        
        x += residual
        x = self.relu(x)
        return x

### Define the ResNet model

In [52]:
class ResNet(models.Model):
    def __init__(self, num_classes=10):
        super(ResNet, self).__init__()
        self.conv = layers.Conv2D(64, kernel_size=3, strides=1, padding='same', use_bias=False)
        self.bn = layers.BatchNormalization()
        self.relu = layers.ReLU()
        self.layer1 = self._make_layer(64, 2)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        self.pool = layers.GlobalAveragePooling2D()
        self.fc = layers.Dense(num_classes, activation='softmax')

    def _make_layer(self, filters, blocks, stride=1):
        downsample = stride != 1
        layers_list = [ResidualBlock(filters, stride=stride, downsample=downsample)]
        for _ in range(1, blocks):
            layers_list.append(ResidualBlock(filters))
        return models.Sequential(layers_list)

    def call(self, inputs, training=False):
        x = self.conv(inputs)
        x = self.bn(x, training=training)
        x = self.relu(x)
        x = self.layer1(x, training=training)
        x = self.layer2(x, training=training)
        x = self.layer3(x, training=training)
        x = self.layer4(x, training=training)
        x = self.pool(x)
        x = self.fc(x)
        return x

### Data preprocessing and augmentation


In [53]:
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

train_datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1,
    brightness_range=[0.8, 1.2],
    preprocessing_function=lambda x: (x / 127.5) - 1
)

test_datagen = ImageDataGenerator(preprocessing_function=lambda x: (x / 127.5) - 1)
train_generator = train_datagen.flow(x_train, y_train, batch_size=64)
test_generator = test_datagen.flow(x_test, y_test, batch_size=64)


### Train final model

In [54]:
device = "cuda" if tf.config.list_physical_devices('GPU') else "cpu"
print(f"Using device: {device}")

model = ResNet(num_classes=10)
model.build(input_shape=(None, 32, 32, 3))
model.summary()

model.compile(
    optimizer=optimizers.SGD(learning_rate=0.001, momentum=0.95, decay=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Using device: cuda


/opt/conda/lib/python3.10/site-packages/keras/src/layers/layer.py:361: UserWarning: `build()` was called on layer 'res_net_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "res_net_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_40 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_40          │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_26 (ReLU)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_7 (Sequential)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_9 (Sequential)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_11 (Sequential)      │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_13 (Sequential)      │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ ?                      │   0 (unbuilt) │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

/opt/conda/lib/python3.10/site-packages/keras/src/optimizers/base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


### Early stopping


In [55]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    train_generator,
    epochs=20,
    validation_data=test_generator,
    callbacks=[early_stopping]
)

Epoch 1/20


/opt/conda/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


782/782 ━━━━━━━━━━━━━━━━━━━━ 73s 73ms/step - accuracy: 0.3515 - loss: 1.7650 - val_accuracy: 0.5313 - val_loss: 1.2770
Epoch 2/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 65ms/step - accuracy: 0.5582 - loss: 1.2246 - val_accuracy: 0.6138 - val_loss: 1.0999
Epoch 3/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 64ms/step - accuracy: 0.6476 - loss: 0.9924 - val_accuracy: 0.6932 - val_loss: 0.8732
Epoch 4/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 50s 64ms/step - accuracy: 0.6966 - loss: 0.8553 - val_accuracy: 0.6424 - val_loss: 1.1398
Epoch 5/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 64ms/step - accuracy: 0.7372 - loss: 0.7524 - val_accuracy: 0.7192 - val_loss: 0.8580
Epoch 6/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 64ms/step - accuracy: 0.7613 - loss: 0.6765 - val_accuracy: 0.7460 - val_loss: 0.7395
Epoch 7/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 65ms/step - accuracy: 0.7814 - loss: 0.6242 - val_accuracy: 0.7825 - val_loss: 0.6455
Epoch 8/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 64ms/step - accuracy: 0.7964 - loss: 0.5773 - val_accurac

### Evaluate best model

In [62]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Accuracy on the test set: {test_acc * 100:.2f}%")

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.5944 - loss: 1.5391
Accuracy on the test set: 59.06%


---
## Transfer learning: ResNet50

### Load CIFAR-10 dataset


In [27]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


### Load pre-trained ResNet50 model

In [28]:
model = models.resnet50(pretrained=True)

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 206MB/s]


### Modify the model for CIFAR-10


In [29]:
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

### Set device

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

### Define loss and optimizer


In [32]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### Training the model


In [33]:
def train_model(model, train_loader, criterion, optimizer, device, epochs=10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

### Evaluate the model


In [34]:
def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Accuracy on the test set: {accuracy:.2f}%")

### Train and evaluate


In [35]:
train_model(model, train_loader, criterion, optimizer, device, epochs=10)
evaluate_model(model, test_loader, device)

/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Epoch [1/10], Loss: 1.1800
Epoch [2/10], Loss: 0.9168
Epoch [3/10], Loss: 0.8245
Epoch [4/10], Loss: 0.7986
Epoch [5/10], Loss: 0.7369
Epoch [6/10], Loss: 0.6528
Epoch [7/10], Loss: 0.6805
Epoch [8/10], Loss: 0.5584
Epoch [9/10], Loss: 0.5483
Epoch [10/10], Loss: 0.5474
Accuracy on the test set: 52.81%


---
# CIFAR100
## Pytorch CNN model trained from scratch


### Data Preprocessing


In [67]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=train_transform)
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


### Training the Model



In [68]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet(num_classes=100).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.95, weight_decay=0.0001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

### Training Loop


In [70]:
epochs = 20
patience = 5
best_loss = float('inf')
stopping_counter = 0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss:.4f}")

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        stopping_counter = 0
        torch.save(model.state_dict(), 'best_resnet_cifar100.pth')
    else:
        stopping_counter += 1

    if stopping_counter >= patience:
        print("Early stopping triggered")
        break


Epoch [1/20], Loss: 3.3056
Epoch [2/20], Loss: 2.8664
Epoch [3/20], Loss: 2.5218
Epoch [4/20], Loss: 2.2688
Epoch [5/20], Loss: 2.0794
Epoch [6/20], Loss: 1.9196
Epoch [7/20], Loss: 1.8011
Epoch [8/20], Loss: 1.7003
Epoch [9/20], Loss: 1.6141
Epoch [10/20], Loss: 1.5308
Epoch [11/20], Loss: 1.2830
Epoch [12/20], Loss: 1.2242
Epoch [13/20], Loss: 1.2006
Epoch [14/20], Loss: 1.1744
Epoch [15/20], Loss: 1.1602
Epoch [16/20], Loss: 1.1483
Epoch [17/20], Loss: 1.1361
Epoch [18/20], Loss: 1.1142
Epoch [19/20], Loss: 1.0996
Epoch [20/20], Loss: 1.0915


### Testing the Model

In [71]:
model.load_state_dict(torch.load('best_resnet_cifar100.pth'))
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy on the CIFAR-100 test set: {accuracy:.2f}%")

/tmp/ipykernel_23/966527642.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_resnet_cifar100.pth'))


Accuracy on the CIFAR-100 test set: 64.83%


---
## Keras CNN model trained from scratch

### Data Preparation

In [58]:
(x_train, y_train), (x_test, y_test) = datasets.cifar100.load_data()
y_train = tf.keras.utils.to_categorical(y_train, 100)
y_test = tf.keras.utils.to_categorical(y_test, 100)

train_datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1,
    brightness_range=[0.8, 1.2],
    preprocessing_function=lambda x: (x / 127.5) - 1
)

test_datagen = ImageDataGenerator(preprocessing_function=lambda x: (x / 127.5) - 1)
train_generator = train_datagen.flow(x_train, y_train, batch_size=64)
test_generator = test_datagen.flow(x_test, y_test, batch_size=64)


### Training the Model

In [59]:
device = "cuda" if tf.config.list_physical_devices('GPU') else "cpu"
print(f"Using device: {device}")

model = ResNet(num_classes=100)
model.build(input_shape=(None, 32, 32, 3))
model.summary()

model.compile(
    optimizer=optimizers.SGD(learning_rate=0.001, momentum=0.95, decay=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Using device: cuda


/opt/conda/lib/python3.10/site-packages/keras/src/layers/layer.py:361: UserWarning: `build()` was called on layer 'res_net_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "res_net_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_60 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_60          │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_35 (ReLU)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_14 (Sequential)      │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_16 (Sequential)      │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_18 (Sequential)      │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_20 (Sequential)      │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ ?                      │   0 (unbuilt) │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Early Stopping

In [60]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    train_generator,
    epochs=20,
    validation_data=test_generator,
    callbacks=[early_stopping]
)

Epoch 1/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 65s 71ms/step - accuracy: 0.0833 - loss: 4.1528 - val_accuracy: 0.1886 - val_loss: 3.3659
Epoch 2/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 64ms/step - accuracy: 0.2219 - loss: 3.2017 - val_accuracy: 0.2680 - val_loss: 2.9508
Epoch 3/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 64ms/step - accuracy: 0.2868 - loss: 2.8499 - val_accuracy: 0.3131 - val_loss: 2.7694
Epoch 4/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 64ms/step - accuracy: 0.3425 - loss: 2.5672 - val_accuracy: 0.3593 - val_loss: 2.5222
Epoch 5/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 64ms/step - accuracy: 0.3902 - loss: 2.3253 - val_accuracy: 0.3867 - val_loss: 2.4014
Epoch 6/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 65ms/step - accuracy: 0.4371 - loss: 2.1255 - val_accuracy: 0.4308 - val_loss: 2.1864
Epoch 7/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 64ms/step - accuracy: 0.4695 - loss: 1.9666 - val_accuracy: 0.4394 - val_loss: 2.1689
Epoch 8/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 64ms/step - accuracy: 0.4995 - loss: 1.8280 - 

### Evaluate on Test Set

In [61]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Accuracy on the test set: {test_acc * 100:.2f}%")

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.5947 - loss: 1.5021
Accuracy on the test set: 59.06%


---
## Transfer learning: ResNet50

### Data Preparation

In [57]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

### Load pre-trained ResNet50 model

In [31]:
model = models.resnet50(pretrained=True)

In [32]:
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 100)

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [34]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [35]:
def train_model(model, train_loader, criterion, optimizer, device, epochs=10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

In [36]:
def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Accuracy on the test set: {accuracy:.2f}%")

In [37]:
train_model(model, train_loader, criterion, optimizer, device, epochs=10)
evaluate_model(model, test_loader, device)

Epoch [1/10], Loss: 3.0959
Epoch [2/10], Loss: 2.2908
Epoch [3/10], Loss: 2.2517
Epoch [4/10], Loss: 2.0502
Epoch [5/10], Loss: 1.8459
Epoch [6/10], Loss: 1.7126
Epoch [7/10], Loss: 1.5142
Epoch [8/10], Loss: 1.4208
Epoch [9/10], Loss: 1.3397
Epoch [10/10], Loss: 1.2960
Accuracy on the test set: 56.99%
